# General Results
This notebook contains results that are not specific to either model but rather concern both models

In [ ]:
import glob
import ast
from collections import defaultdict

import pandas as pd
import numpy as np

from src.evaluation import get_mean_and_ci

In [ ]:
tag_labels = [
    "<INTER>",
    "<CORR>",
    "<ERR_DESC>",
    "<INST>",
    "<ERR_SIM>",
    "<PLAUS>",
    "<CURATE>",
    "<RECON>"
]

tag_names = [lbl.strip("<>").lower() for lbl in tag_labels]


TAG_DISPLAY_NAMES = {
    "inter": "Task Interpretation",
    "corr": "Correct Answer Ref.",
    "err_desc": "Error Description",
    "inst": "Outcome Instantiation",
    "err_sim": "Error Simulation",
    "plaus": "Plausibility Check",
    "curate": "Final Set Curation",
    "recon": "Reconsideration"
}

In [ ]:
# Load and Parse Annotated Data
def load_and_parse_traces(pattern):
    """
    Load annotated traces from CSV files matching the pattern.
    Returns a dictionary with tag names as keys and lists of normalized positions as values.
    """
    positions_by_tag = {tag: [] for tag in tag_names}
    
    parsed_csv_paths = glob.glob(pattern)
    
    for path in parsed_csv_paths:
        df = pd.read_csv(path)
        for idx, row in df.iterrows():
            reasoning = row.get("trace", "")
            seq_str = row.get("annotation_sequence", "")
            
            seq = ast.literal_eval(seq_str) if isinstance(seq_str, str) and seq_str.startswith("[") else []
            reasoning_len = len(reasoning) if reasoning else 1
            
            for pos, tag in seq:
                if tag in positions_by_tag:
                    norm_pos = pos / reasoning_len
                    positions_by_tag[tag].append(norm_pos)
    
    return positions_by_tag

# Load reasoning traces
print("Loading reasoning traces...")
reasoning_positions = load_and_parse_traces(f"eedi_data/joint_results/annotated/*-reasoner_*_annot_parsed.csv")
print(f"Loaded reasoning traces with {sum(len(v) for v in reasoning_positions.values())} total component occurrences")

# Load CoT traces
print("Loading CoT traces...")
cot_positions = load_and_parse_traces(f"eedi_data/joint_results/annotated/*-cot-*-chat_*_annot_parsed.csv")
print(f"Loaded CoT traces with {sum(len(v) for v in cot_positions.values())} total component occurrences")

### Summary Statistics

In [ ]:
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

# Helper: compute trace-level stats (avg length, presence counts, counts per trace)
from collections import Counter

def compute_trace_stats(pattern):
    paths = glob.glob(pattern)
    total_traces = 0
    total_len = 0
    presence = {tag: 0 for tag in tag_names}
    counts_per_trace = {tag: [] for tag in tag_names}

    for path in paths:
        df = pd.read_csv(path)
        for _, row in df.iterrows():
            trace = row.get("trace", "")
            seq_str = row.get("annotation_sequence", "")
            seq = ast.literal_eval(seq_str) if isinstance(seq_str, str) and seq_str.startswith("[") else []
            total_traces += 1
            total_len += len(trace) if isinstance(trace, str) else 0

            labels_list = [label for _, label in seq if label in tag_names]
            labels_in_trace = set(labels_list)

            for tag in labels_in_trace:
                presence[tag] += 1

            counts = Counter(labels_list)
            for tag in tag_names:
                counts_per_trace[tag].append(counts.get(tag, 0))

    avg_len = (total_len / total_traces) if total_traces > 0 else 0
    presence_pct = {tag: (presence[tag] / total_traces * 100) if total_traces > 0 else 0 for tag in tag_names}
    return total_traces, avg_len, presence, presence_pct, counts_per_trace


# 2) Compute average lengths
reasoning_pattern = f"eedi_data/joint_results/annotated/*-reasoner_*_annot_parsed.csv"
cot_pattern = f"eedi_data/joint_results/annotated/*-cot-*-chat_*_annot_parsed.csv"

r_total, r_avg_len, r_presence, r_presence_pct, r_counts_per_trace = compute_trace_stats(reasoning_pattern)
ct_total, ct_avg_len, ct_presence, ct_presence_pct, ct_counts_per_trace = compute_trace_stats(cot_pattern)

print("\nAverage trace lengths:")
print(f"  Reasoning traces (n={r_total}): avg length = {r_avg_len:.1f} chars")
print(f"  CoT traces (n={ct_total}): avg length = {ct_avg_len:.1f} chars")

# 2b) Average occurrences per trace (mean ± 95% CI)
print("\nAverage occurrences per trace (mean ± 95% CI):")
print("  Reasoning traces:")
for tag in tag_names:
    arr = np.array(r_counts_per_trace[tag])
    if len(arr) > 1:
        mean, ci = get_mean_and_ci(arr)
    else:
        mean, ci = (np.mean(arr) if len(arr) > 0 else 0), 0
    print(f"    {TAG_DISPLAY_NAMES.get(tag, tag)}: {mean:.2f} ± {ci:.2f}")
print("  CoT traces:")
for tag in tag_names:
    arr = np.array(ct_counts_per_trace[tag])
    if len(arr) > 1:
        mean, ci = get_mean_and_ci(arr)
    else:
        mean, ci = (np.mean(arr) if len(arr) > 0 else 0), 0
    print(f"    {TAG_DISPLAY_NAMES.get(tag, tag)}: {mean:.2f} ± {ci:.2f}")

# 3) Percentages of traces containing at least one occurrence per label
print("\nPercentage of traces containing at least one occurrence (per label):")
print("  Reasoning traces:")
for tag in tag_names:
    pct = r_presence_pct.get(tag, 0)
    print(f"    {TAG_DISPLAY_NAMES.get(tag, tag)}: {pct:.1f}%")
print("  CoT traces:")
for tag in tag_names:
    pct = ct_presence_pct.get(tag, 0)
    print(f"    {TAG_DISPLAY_NAMES.get(tag, tag)}: {pct:.1f}%")

print("\n" + "="*80)

### Similarity Between Models

In [ ]:
# Top-k dominant agreement (row-wise, Sankey-style)
print("\n" + "="*80)
print("ROW-WISE TOP-K DOMINANT AGREEMENT (≥15%)")
print("="*80)

def get_row_wise_transitions(pattern):
    """
    Build a row-wise (source -> target) Counter for all transitions across traces.
    Returns dict: source_tag -> Counter(target_tag -> count)
    """
    row_transitions = defaultdict(Counter)
    
    parsed_csv_paths = glob.glob(pattern)
    
    for path in parsed_csv_paths:
        df = pd.read_csv(path)
        for _, row in df.iterrows():
            seq_str = row.get("annotation_sequence", "")
            seq = ast.literal_eval(seq_str) if isinstance(seq_str, str) and seq_str.startswith("[") else []
            
            tags = [tag for _, tag in seq if tag in tag_names]
            
            for i in range(len(tags) - 1):
                src, tgt = tags[i], tags[i+1]
                row_transitions[src][tgt] += 1
    
    return row_transitions

def top_k_rowwise_dominant_agreement(trans1, trans2, k=2, threshold=0.15):
    """
    Compute top-k dominant agreement per source node (row-wise).
    Only count transitions whose fraction of outgoing flow >= threshold.
    Returns average agreement across all source nodes.
    """
    source_tags = set(list(trans1.keys()) + list(trans2.keys()))
    overlaps = []
    
    for src in source_tags:
        counter1 = trans1.get(src, Counter())
        counter2 = trans2.get(src, Counter())
        
        total1 = sum(counter1.values()) + 1e-10
        total2 = sum(counter2.values()) + 1e-10
        
        # Filter transitions by row-wise threshold
        top1 = [t for t, c in counter1.items() if c / total1 >= threshold]
        top2 = [t for t, c in counter2.items() if c / total2 >= threshold]
        
        # Take top-k from the filtered set
        top1 = top1[:k]
        top2 = top2[:k]
        
        if len(top1) == 0 and len(top2) == 0:
            continue  # skip rows with no dominant transitions
        elif len(top1) == 0 or len(top2) == 0:
            overlaps.append(0.0)
        else:
            overlap = len(set(top1) & set(top2)) / min(len(top1), len(top2))
            overlaps.append(overlap)
    
    if len(overlaps) == 0:
        return np.nan
    return np.mean(overlaps)

# Build row-wise transition counters
glm_reasoning_transitions = get_row_wise_transitions("eedi_data/joint_results/annotated/*openrouter*reasoner*_annot_parsed.csv")
glm_cot_transitions = get_row_wise_transitions("eedi_data/joint_results/annotated/*openrouter*cot*chat*_annot_parsed.csv")
deepseek_reasoning_transitions = get_row_wise_transitions("eedi_data/joint_results/annotated/*deepseek*reasoner*_annot_parsed.csv")
deepseek_cot_transitions = get_row_wise_transitions("eedi_data/joint_results/annotated/*deepseek*cot*chat*_annot_parsed.csv")

k = 3
min_outgoing_mass = 0.15
print("\n=== REASONING TRACES ===")
reasoning_agreement = top_k_rowwise_dominant_agreement(glm_reasoning_transitions, deepseek_reasoning_transitions, k=k, threshold=min_outgoing_mass)
print(f"Row-wise top-{k} dominant agreement (>={min_outgoing_mass}): {reasoning_agreement:.2f}")

print("\n=== CoT TRACES ===")
cot_agreement = top_k_rowwise_dominant_agreement(glm_cot_transitions, deepseek_cot_transitions, k=k, threshold=min_outgoing_mass)
print(f"Row-wise top-{k} dominant agreement (>={min_outgoing_mass}): {cot_agreement:.2f}")


In [4]:
import numpy as np

strategies = [
    "Task Interpretation",
    "Correct Answer Ref.",
    "Error Description",
    "Outcome Instantiation",
    "Error Simulation",
    "Plausibility Check",
    "Final Set Curation",
    "Reconsideration"
]

ds_cot_means = np.array([
    2.34,
    4.33,
    7.33,
    7.32,
    2.02,
    1.48,
    0.63,
    0.86
])

ds_reasoning_means = np.array([
    11.37,
    10.25,
    21.58,
    37.43,
    6.10,
    8.63,
    3.36,
    7.36
])


glm_cot_means = np.array([
    1.11,
    4.09,
    9.02,
    8.80,
    4.74,
    1.43,
    0.64,
    0.90
])

glm_reasoning_means = np.array([
    4.36,
    11.47,
    19.54,
    32.51,
    7.19,
    4.67,
    3.46,
    2.54
])

# Ratio (Reasoning / CoT)
ds_ratio = ds_reasoning_means / ds_cot_means
glm_ratio = glm_reasoning_means / glm_cot_means

for s, d, g in zip(strategies, ds_ratio, glm_ratio):
    print("DS vs GLM")
    print(f"{s}: {d:.2f} vs {g:.2f} ({2*abs(d-g)/(d+g)})")

DS vs GLM
Task Interpretation: 4.86 vs 3.93 (0.2119168736937576)
DS vs GLM
Correct Answer Ref.: 2.37 vs 2.80 (0.1690752896680339)
DS vs GLM
Error Description: 2.94 vs 2.17 (0.30438872048580573)
DS vs GLM
Outcome Instantiation: 5.11 vs 3.69 (0.32223368276634196)
DS vs GLM
Error Simulation: 3.02 vs 1.52 (0.6625657837183283)
DS vs GLM
Plausibility Check: 5.83 vs 3.27 (0.5640098688482016)
DS vs GLM
Final Set Curation: 5.33 vs 5.41 (0.013579049466537398)
DS vs GLM
Reconsideration: 8.56 vs 2.82 (1.00803778211707)


In [5]:
ds_cot_means.mean() - glm_cot_means.mean(), ds_reasoning_means.mean() - glm_reasoning_means.mean()

(np.float64(-0.5524999999999998), np.float64(2.5425000000000004))